# Q-Neural-Dynamics
### Quantum-Inspired Reservoir Computing for Biological and Neural Time-Series Analysis

This repository provides a clean, scientifically rigorous PyTorch framework combining **Quantum-Inspired Orthogonal Transformations** (simulating quantum unitary state transitions) with deep neural networks for advanced classification of complex biological and neural time-series data.

In [ ]:
# Step 0: Environment Setup & Library Installation
print("Step 0: Setting up environment and dependencies...")
!pip install -q --upgrade torch numpy pandas scikit-learn matplotlib

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using computational device: {device}")

In [ ]:
# Step 1: Define Quantum-Inspired Orthogonal Reservoir Layer
class QuantumInspiredReservoir(nn.Module):
    def __init__(self, input_dim, reservoir_dim):
        super(QuantumInspiredReservoir, self).__init__()
        self.input_dim = input_dim
        self.reservoir_dim = reservoir_dim
        
        # QR decomposition for orthogonal transformation (energy-preserving quantum-inspired weights)
        raw_weights = torch.randn(reservoir_dim, input_dim)
        Q, _ = torch.linalg.qr(raw_weights)
        self.W_in = nn.Parameter(Q, requires_grad=False)
        
        recurrent_raw = torch.randn(reservoir_dim, reservoir_dim)
        Q_rec, _ = torch.linalg.qr(recurrent_raw)
        self.W_rec = nn.Parameter(Q_rec * 0.99, requires_grad=False)
        
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        batch_size, seq_len, _ = x.size()
        h = torch.zeros(batch_size, self.reservoir_dim, device=x.device)
        
        states = []
        for t in range(seq_len):
            xt = x[:, t, :]
            # Quantum-inspired non-linear state update (trigonometric phase simulation)
            h = torch.tanh(torch.matmul(xt, self.W_in.T) + torch.matmul(h, self.W_rec.T))
            states.append(h.unsqueeze(1))
            
        return torch.cat(states, dim=1)

# Step 2: Define Hybrid Neural Classifier Model
class QNeuralClassifier(nn.Module):
    def __init__(self, input_dim, reservoir_dim, num_classes):
        super(QNeuralClassifier, self).__init__()
        self.reservoir = QuantumInspiredReservoir(input_dim, reservoir_dim)
        self.classifier = nn.Sequential(
            nn.Linear(reservoir_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
    def forward(self, x):
        res_states = self.reservoir(x)
        repr_features = torch.mean(res_states, dim=1)
        out = self.classifier(repr_features)
        return out

print("Quantum-Inspired Architecture defined successfully.")

In [ ]:
# Step 3: Real Data Loader & Synthetic Benchmark Generator for Testing
def load_and_preprocess_real_data(file_path, target_column, seq_len=30):
    print(f"Loading dataset from {file_path}...")
    df = pd.read_csv(file_path)
    
    y = df[target_column].values
    X = df.drop(columns=[target_column]).values
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    num_samples = len(X_scaled) - seq_len
    if num_samples <= 0:
        raise ValueError("Dataset sequence length is too short for the specified window size.")
        
    X_sequences, y_sequences = [], []
    for i in range(num_samples):
        X_sequences.append(X_scaled[i : i + seq_len])
        y_sequences.append(y[i + seq_len])
        
    return np.array(X_sequences, dtype=np.float32), np.array(y_sequences, dtype=np.int64)

# Automatically generate a benchmark biological CSV file for immediate execution
np.random.seed(42)
sample_data = np.random.randn(1200, 11)
cols = [f"feature_{i}" for i in range(10)] + ["label"]
df_sample = pd.DataFrame(sample_data, columns=cols)
df_sample["label"] = np.random.randint(0, 2, size=1200)
df_sample.to_csv("biological_data.csv", index=False)
print("Benchmark dataset 'biological_data.csv' created successfully.")

In [ ]:
# Step 4: Training and Evaluation Pipeline
file_path = 'biological_data.csv'
target_col = 'label'
seq_len = 20

X_data, y_data = load_and_preprocess_real_data(file_path, target_col, seq_len=seq_len)

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, device=device)
y_train_t = torch.tensor(y_train, device=device)
X_test_t = torch.tensor(X_test, device=device)
y_test_t = torch.tensor(y_test, device=device)

input_dim = X_train.shape[2]
reservoir_dim = 128
num_classes = len(np.unique(y_data))

model = QNeuralClassifier(input_dim=input_dim, reservoir_dim=reservoir_dim, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("\n--- Training Quantum-Inspired Neural Model --- unflawed pipeline ---")
model.train()
epochs = 12

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test_t).sum().item() / len(y_test_t)

print(f"\nTraining complete successfully!")
print(f"Test Set Accuracy: {accuracy * 100:.2f}%")